# YOLOv26 Training Pipeline (Google Colab)
Notebook ini digunakan untuk training model YOLOv26 menggunakan GPU dari Google Colab.

**Fitur baru YOLOv26:**
- NMS-Free End-to-End inference (tanpa post-processing)
- MuSGD Optimizer (hybrid SGD + Muon dari Kimi K2)
- Hingga 43% lebih cepat di CPU dibanding YOLO11
- DFL Removal untuk kompatibilitas edge device

In [ ]:
# Cek GPU yang tersedia
!nvidia-smi

In [ ]:
# Install ultralytics (harus versi terbaru untuk support YOLOv26)
!pip install -q ultralytics roboflow

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
from ultralytics import YOLO
from google.colab import drive
print("Ultralytics version:", __import__('ultralytics').__version__)

In [ ]:
# Mount Drive untuk menyimpan hasil training
drive.mount('/content/drive')

# Membuat folder output di Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/yolov26_results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
from roboflow import Roboflow

# sesuaikan dengan info dari Roboflow user
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(YOUR_VERSION_NUMBER)
dataset = version.download("yolov8")  # format YOLO ini tetap kompatibel dengan v26

data_yaml = os.path.join(dataset.location, 'data.yaml')
print(f"Dataset location: {dataset.location}")
print(f"data.yaml: {data_yaml}")

In [ ]:
# Verifikasi dataset
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files[:10]:
        print(f'{subindent}{f}')

## Konfigurasi Training
Ubah parameter sesuai dengan kebutuhan

In [ ]:
# ============================
# KONFIGURASI TRAINING
# ============================

# Model YOLOv26
# yolo26n.pt = Nano (Tercepat, cocok edge device)
# yolo26s.pt = Small
# yolo26m.pt = Medium
# yolo26l.pt = Large
# yolo26x.pt = Extra Large (Paling akurat)
MODEL_NAME = 'yolo26n.pt'  # sesuaikan sendiri

# Training
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640
PATIENCE = 20       # Early Stopping
DEVICE = 0          # 0 = GPU PERTAMA

# Nama Project
PROJECT_NAME = 'runs/yolov26'
EXPERIMENT_NAME = 'train_colab'

In [ ]:
# Load pretrained model
model = YOLO(MODEL_NAME)

# Jalankan Training
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    plots=True,
    save=True,
    val=True,
)
print(results)

In [ ]:
from IPython.display import Image, display

# Tampilkan grafik hasil training
results_dir = f'{PROJECT_NAME}/{EXPERIMENT_NAME}'
for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = os.path.join(results_dir, img_name)
    if os.path.exists(img_path):
        print(f'\n--- {img_name} ---')
        display(Image(filename=img_path, width=800))

In [ ]:
# Validasi pada validation set
best_model = YOLO(f'{results_dir}/weights/best.pt')
metrics = best_model.val(data=data_yaml, split='val', device=DEVICE)

print(f"\nmAP@50     : {metrics.box.map50:.4f}")
print(f"mAP@50-95  : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

In [ ]:
import shutil

# Copy best.pt dan last.pt ke Google Drive
weights_dir = f'{results_dir}/weights'
for weight_file in ['best.pt', 'last.pt']:
    src = os.path.join(weights_dir, weight_file)
    dst = os.path.join(DRIVE_OUTPUT, weight_file)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"{weight_file} disalin ke: {dst}")

# Copy folder results lengkap
results_drive = os.path.join(DRIVE_OUTPUT, EXPERIMENT_NAME)
if os.path.exists(results_dir):
    shutil.copytree(results_dir, results_drive, dirs_exist_ok=True)
    print(f"Seluruh hasil disalin ke: {results_drive}")

In [ ]:
# (Opsional) Export ke ONNX
exported = best_model.export(format='onnx', imgsz=IMG_SIZE)
print(f"Exported to: {exported}")

# Copy ke Drive
if exported and os.path.exists(str(exported)):
    shutil.copy2(str(exported), DRIVE_OUTPUT)
    print(f"ONNX disalin ke Drive")